%spark.pyspark
from pyspark.sql import functions as F

users = spark.createDataFrame(
    [
        ("u1", "Berlin"),
        ("u2", "Berlin"),
        ("u3", "Munich"),
        ("u4", "Hamburg"),
    ],
    ["user_id", "city"]
)

products = spark.createDataFrame(
    [
        ("p1", "Ring VOLA"),
        ("p2", "Ring POROG"),
        ("p3", "Ring TISHINA"),
    ],
    ["product_id", "product_name"]
)

orders = spark.createDataFrame(
    [
        ("o1", "u1", "p1", 2, 10.0),
        ("o2", "u1", "p2", 1, 30.0),
        ("o3", "u2", "p1", 1, 10.0),
        ("o4", "u2", "p3", 5, 7.0),
        ("o5", "u3", "p2", 3, 30.0),
        ("o6", "u3", "p3", 1, 7.0),
        ("o7", "u4", "p1", 10, 10.0),
    ],
    ["order_id", "user_id", "product_id", "qty", "price"]
)

%spark.pyspark
users.show()

%spark.pyspark
products.show()

%spark.pyspark
orders.show()

Задание 1

%spark.pyspark
orders = orders.withColumn("revenue", orders["qty"] * orders["price"])

%spark.pyspark
orders.show()

Задание 2

%spark.pyspark
full_data = broadcast(orders).join(users, "user_id").join(products, "product_id")

%spark.pyspark
full_data.show()

Задание 3

%spark.pyspark
orders_cnt = full_data.groupBy("city").agg(F.count("order_id"))
qty_sum = full_data.groupBy("city").agg(F.sum("qty"))
revenue_sum = full_data.groupBy("city").agg(F.sum("revenue"))
metrics_by_city = broadcast(orders_cnt).join(qty_sum, "city").join(revenue_sum, "city")
metrics_by_city.show()

%spark.pyspark
orders_cnt = full_data.groupBy("product_id").agg(F.count("order_id"))
qty_sum = full_data.groupBy("product_id").agg(F.sum("qty"))
revenue_sum = full_data.groupBy("product_id").agg(F.sum("revenue"))
metrics_by_product_id = broadcast(orders_cnt).join(qty_sum, "product_id").join(revenue_sum, "product_id")
metrics_by_product_id.show()

%spark.pyspark
orders_cnt = full_data.groupBy("product_name").agg(F.count("order_id"))
qty_sum = full_data.groupBy("product_name").agg(F.sum("qty"))
revenue_sum = full_data.groupBy("product_name").agg(F.sum("revenue"))
metrics_by_product_name = broadcast(orders_cnt).join(qty_sum, "product_name").join(revenue_sum, "product_name")
metrics_by_product_name.show()

Задание 4

%spark.pyspark
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, sum, col

top_revenue = full_data.groupBy("city", "product_name").agg(sum("revenue"))

w = Window.partitionBy("city").orderBy(desc("sum(revenue)"))
top_revenue = top_revenue.withColumn("rn", row_number().over(w)).filter("rn <= 2")

top_revenue.show()

Задание 5 и 6

%spark.pyspark
path = "s3a://for-dataproc-tat/mart_city_top_products/"
top_revenue.write.mode("overwrite").parquet(path)
spark.read.parquet(path).show()

%spark.pyspark
path = "/tmp/sandbox_zeppelin/mart_city_top_products/"
top_revenue.write.mode("overwrite").parquet(path)
spark.read.parquet(path).show()